> **Multi-node workspace required:** This is a multi-node example. To run it,
> your Modal workspace must have multi-node enabled. Contact
> [support@modal.com](mailto:support@modal.com) to enable multi-node.

# Multi-node Qwen3.6-27B full-weight training

This tutorial runs full-weight GRPO on
[Qwen3.6-27B](https://huggingface.co/Qwen/Qwen3.6-27B), a
27B-parameter hybrid language model from Qwen, using
[slime](https://github.com/THUDM/slime) across **4 nodes
(32 H100 GPUs)**.

The `Qwen3_6_27b_Recipe` preset configures colocated training and
rollout workers, EAGLE speculative decoding, CPU-offloaded Adam,
and DeepScaler math reward verification.

## Prerequisites

This tutorial requires a Modal Secret named `huggingface-secret` containing your
`HF_TOKEN`. Create one at [modal.com/secrets](https://modal.com/secrets) if you
haven't already — the cell below fails fast with instructions otherwise.

> **Note:** you do **not** need to attach a GPU to this notebook. All training and
> serving happens on Modal-managed GPU workers spun up by the SDK — the notebook
> itself only needs to issue API calls.

In [ ]:
import modal

try:
    modal.Secret.from_name("huggingface-secret").hydrate()
except modal.exception.NotFoundError as e:
    raise RuntimeError(
        "Missing Modal Secret 'huggingface-secret'. Create one at "
        "https://modal.com/secrets with an HF_TOKEN entry, then re-run."
    ) from e

In [ ]:
import importlib.util

# Skip if modal_training_gym is already importable (e.g. a local editable
# checkout) so your edits keep taking effect and the env stays synced.
if importlib.util.find_spec('modal_training_gym') is None:
    %uv pip install -q git+https://github.com/modal-projects/training-gym.git@main

In [ ]:
import modal

from modal_training_gym.common.modal_urls import modal_app_dashboard_url
from modal_training_gym import (
    HuggingFaceDataset,
    Qwen3_6_27B,
    Qwen3_6_27b_Recipe,
    TrainConfig,
)

## Dataset

We use [DAPO-Math-17k](https://huggingface.co/datasets/zhuzilin/dapo-math-17k),
a collection of math competition problems with verifiable answers.
The `deepscaler` reward model checks whether the model's response
matches the reference answer.

In [ ]:
class MathDataset(HuggingFaceDataset):
    hf_repo = "zhuzilin/dapo-math-17k"
    input_column = ""
    output_column = ""
    input_key = "prompt"
    label_key = "label"
    output_format = "jsonl"
    apply_chat_template = True
    always_prepare = True

## Build and launch training

Build the training config, construct the Modal app, and spawn
the training function as a detached call.

In [ ]:
def build_training_config() -> TrainConfig:
    return TrainConfig(
        model=Qwen3_6_27B(),
        dataset=MathDataset(n_rows=10),
        recipe=Qwen3_6_27b_Recipe(),
    )

training_run = build_training_config()
app = training_run._build_app()

with modal.enable_output():
    # detach=True keeps the app (and the spawned train call) alive on Modal
    # after this script exits — a plain app.run() stops the app on exit,
    # which cancels the spawned call remotely.
    with app.run(detach=True):
        modal_app_id = app.app_id or ""
        function_call = app.train.spawn(
            modal_app_id=modal_app_id,
            modal_app_url=modal_app_dashboard_url(modal_app_id),
        )
        print(f"Spawned train function call: {function_call.object_id}")